# Creador de Clips Virales - Acertijos para IG

---
## Conectar con Colab sin borrar el proyecto (usar GitHub)

1. **En tu PC:** sube cambios con `git add .`, `git commit -m "..."`, `git push` (repo: https://github.com/yoquelvisdev08/acertijo).
2. **En Colab:** Menu **File > Open notebook** > pestaña **GitHub** > pega `https://github.com/yoquelvisdev08/acertijo` o busca "acertijo".
3. Abre **creador_acertijos_ig.ipynb** desde la lista. Colab carga el notebook desde GitHub.
4. **Cuando cambies algo en tu PC:** haz `git add .`, `git commit -m "..."`, `git push`. Luego en Colab: **File > Open notebook > GitHub** y vuelve a abrir este notebook; tendras la version nueva. No hace falta borrar ni recrear nada.

**Opcional:** Si prefieres clonar el repo en Colab, ejecuta la **Celda 0** (abajo); luego abre este archivo desde `/content/acertijo/creador_acertijos_ig.ipynb`. Para traer cambios: ejecuta otra vez la Celda 0 (hace `git pull`) y reabre el notebook.

---

1. **Runtime > Change runtime type > GPU** (T4 o superior).
2. Ejecuta las celdas en orden.
3. **Opcional pero recomendado:** Ejecuta la celda **8b (Pre-descarga de modelos)** antes de la celda que lanza la interfaz. Asi los modelos TTS y Wan se descargan una vez y quedan en cache; la primera generacion no se bloquea por la descarga.
4. TTS va en **CPU** (evita CUDA assert en Colab). El **video** usa GPU. Si quieres probar TTS en GPU, en Celda 2 pon `USE_CPU_FOR_TTS = False` (puede dar assert).
5. En la interfaz escribe el texto del acertijo y pulsa **Generar**.
6. Descarga el MP4 desde el reproductor.

**Tiempo por video:** Primera vez 8-15 min (descarga voz + video). Siguientes ~6-11 min. No cierres mientras diga Processing.

**RAM/GPU:** Voz en CPU; video con ModelScope 1.7B en GPU (T4). En Celda 2 debe salir "GPU detectada: Tesla T4". Revisa la **Consola (logs)** debajo de Estado.

**Sobre "This share link expires in 1 week":** Es normal en Colab gratis. El enlace publico dura 7 dias. Para hosting permanente con GPU: `gradio deploy` en Hugging Face Spaces (https://huggingface.co/spaces).

In [1]:
import sys
print('Instalando MoviePy 2.x (evita errores con decorator en Colab)...')
!{sys.executable} -m pip install -q "moviepy>=2.0.0"
print('MoviePy instalado.')

Reinstalando moviepy para asegurar que el modulo este disponible...
Instalacion de moviepy completada.


In [2]:
try:
    from moviepy import VideoFileClip, AudioFileClip, CompositeVideoClip, TextClip
    print('MoviePy 2 listo.')
except Exception as e:
    print(f'Error: {e}. Ejecuta la celda anterior y luego Runtime > Restart runtime.')

Verificando importacion de moviepy.editor...
Error al importar moviepy.editor: No module named 'moviepy.editor'
Por favor, reinicia el entorno de ejecucion (Runtime > Restart runtime) y ejecuta las celdas 1, 2, 3, 4, 5, 6, 7, 8, 8b y luego intenta de nuevo.


In [3]:
# Celda 0 (opcional): Clonar o actualizar desde GitHub para tener siempre la ultima version
# Pon la URL de tu repo y ejecuta esta celda. Luego abre este .ipynb desde la carpeta clonada.
REPO_URL = 'https://github.com/yoquelvisdev08/acertijo.git'
REPO_DIR = '/content/acertijo'

import os
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
    print('Repo actualizado. Si cambiaste el notebook, reabrelo desde File > Open notebook.')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f'Clonado en {REPO_DIR}. Abre el notebook desde ahi: File > Open notebook > buscar creador_acertijos_ig.ipynb')

hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.
Repo actualizado. Si cambiaste el notebook, reabrelo desde File > Open notebook.


In [4]:
# Celda 1: Instalacion (ejecutar una vez)
# requirements_colab.txt (integrado):
#   qwen-tts>=0.1.0, soundfile
#   diffusers>=0.34.0, transformers>=4.45.0, accelerate, ftfy
#   moviepy>=2.0.0, pysrt
#   gradio>=4.0.0
#   torch/torchvision ya en Colab
!apt-get update -qq && apt-get install -qq -y imagemagick sox > /dev/null
!pip install -q "qwen-tts>=0.1.0" soundfile "diffusers>=0.34.0" "transformers>=4.45.0" accelerate ftfy "moviepy==1.0.3" pysrt "gradio>=4.0.0"
print('Dependencias instaladas. (MoviePy 1.0.3 para evitar error moviepy.editor en Colab.)')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencias instaladas.


In [5]:
# Celda 2: Imports y configuracion (estable en Colab T4)
import os
# Evitar bloqueos que suben uso de RAM; dejar en 0 para inferencia
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
# Reducir fragmentacion de VRAM en T4 (opcional pero recomendado)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
import numpy as np
import soundfile as sf
import re
from pathlib import Path

OUTPUT_DIR = Path('/content/salida_acertijos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Comprobar GPU al inicio (Runtime > Change runtime type > GPU / T4)
if not torch.cuda.is_available():
    print('AVISO: No hay GPU. El video usara CPU y puede llenar la RAM y reiniciar el kernel.')
    print('Solucion: Runtime > Change runtime type > Hardware accelerator > GPU (T4).')
else:
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}. VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def sanitize_script(text: str, max_len: int = 500) -> str:
    if not text or not isinstance(text, str):
        return ''
    s = text.strip()
    s = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', s)
    return s[:max_len] if len(s) > max_len else s

def sanitize_video_prompt(text: str, max_len: int = 200) -> str:
    if not text or not isinstance(text, str):
        return 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'
    s = text.strip()
    s = re.sub(r'[^a-zA-Z0-9\s,\.\-]', '', s)
    return (s[:max_len] if len(s) > max_len else s) or 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_CPU_FOR_TTS = True
print(f'Dispositivo: {DEVICE}. TTS en CPU. Video en GPU (obligatorio en T4 para no saturar RAM).')

GPU detectada: Tesla T4. VRAM: 15.6 GB
Dispositivo: cuda. TTS en CPU. Video en GPU (obligatorio en T4 para no saturar RAM).


In [6]:
# Celda 3: Voz con Qwen3-TTS (espanol). Por defecto usa GPU; USE_CPU_FOR_TTS=True en Celda 2 para forzar CPU si hay CUDA assert.
def get_tts_model():
    import time
    print('[TTS] Estamos en: descargando/cargando modelo de VOZ (Qwen3-TTS). Este modelo narra el acertijo.', flush=True)
    t0 = time.time()
    from qwen_tts import Qwen3TTSModel
    tts_device = 'cpu' if USE_CPU_FOR_TTS else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[TTS] Dispositivo: {tts_device} (GPU = menos RAM, mas rapido). Descargando pesos si es la primera vez...', flush=True)
    model = Qwen3TTSModel.from_pretrained(
        'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',
        device_map=tts_device,
        torch_dtype=torch.float16 if tts_device == 'cuda' else torch.float32,
    )
    print(f'[TTS] Modelo de voz listo en {time.time()-t0:.1f}s', flush=True)
    return model

SPEAKERS_VALID = ('Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee', 'Uncle_Fu', 'Dylan', 'Eric')

def generate_voice(text: str, out_path: str, speaker: str = 'Ryan', model=None):
    import time
    t0 = time.time()
    print('[TTS] Iniciando generacion de voz...', flush=True)
    if model is None:
        model = get_tts_model()
    text = sanitize_script(text, max_len=400)
    if not text:
        raise ValueError('Texto del acertijo vacio tras sanitizar.')
    speaker = speaker if speaker in SPEAKERS_VALID else 'Ryan'
    wavs, sr = model.generate_custom_voice(
        text=text,
        language='Spanish',
        speaker=speaker,
    )
    sf.write(out_path, wavs[0], sr)
    dur = len(wavs[0]) / sr
    print(f'[TTS] Listo en {time.time()-t0:.1f}s (audio {dur:.1f}s)', flush=True)
    return out_path, sr, dur

In [7]:
# Celda 4: Video con ModelScope text-to-video-ms-1.7b (usa GPU en Colab T4, mas estable que Wan)
def get_video_pipeline():
    import time
    import gc
    if not torch.cuda.is_available():
        raise RuntimeError('Se necesita GPU (T4). Runtime > Change runtime type > GPU.')
    from diffusers import DiffusionPipeline
    from diffusers.utils import export_to_video
    model_id = 'damo-vilab/text-to-video-ms-1.7b'
    print('[VIDEO] Cargando modelo de VIDEO en GPU (ModelScope 1.7B)...', flush=True)
    t0 = time.time()
    gc.collect()
    torch.cuda.empty_cache()
    pipeline = DiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        variant='fp16',
    )
    pipeline = pipeline.to('cuda')
    pipeline.enable_vae_slicing()
    gc.collect()
    torch.cuda.empty_cache()
    print(f'[VIDEO] Pipeline en GPU ({time.time()-t0:.1f}s). VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB', flush=True)
    return pipeline

def generate_video(prompt: str, out_path: str, num_frames: int = 24, pipeline=None):
    import time
    import gc
    from diffusers.utils import export_to_video
    if pipeline is None:
        pipeline = get_video_pipeline()
    prompt = sanitize_video_prompt(prompt, max_len=150)
    num_frames = min(max(num_frames, 16), 64)
    print('[VIDEO] Generando frames en GPU (1-3 min)...', flush=True)
    t0 = time.time()
    out = pipeline(
        prompt,
        num_frames=num_frames,
        num_inference_steps=25,
    ).frames[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print(f'[VIDEO] Frames en {time.time()-t0:.1f}s. Exportando MP4...', flush=True)
    export_to_video(out, out_path, fps=8)
    del out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('[VIDEO] Video guardado.', flush=True)
    return out_path

In [8]:
# Celda 5: Subtitulos y composicion (MoviePy)
def make_srt_from_script(script: str, duration_sec: float) -> str:
    lines = [s.strip() for s in script.replace('.', '. ').split('.') if s.strip()]
    if not lines:
        lines = [script]
    n = len(lines)
    step = duration_sec / n if n else duration_sec
    srt = []
    for i, line in enumerate(lines):
        start = i * step
        end = (i + 1) * step
        srt.append(f'{i+1}\n{_ts(start)} --> {_ts(end)}\n{line}\n')
    return '\n'.join(srt)

def _ts(sec: float) -> str:
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int((sec % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def compose_video(video_path: str, audio_path: str, script: str, out_path: str, duration_audio: float):
    import time
    import gc
    from moviepy import VideoFileClip, AudioFileClip, CompositeVideoClip, TextClip
    try:
        from moviepy.config import change_settings
    except ImportError:
        change_settings = lambda x: None
    print('[COMPOSE] Uniendo video + audio + subtitulos (resolucion reducida para ahorrar RAM)...', flush=True)
    t0 = time.time()
    try:
        import shutil
        imagick = shutil.which('convert') or '/usr/bin/convert'
        change_settings({'IMAGEMAGICK_BINARY': imagick})
    except Exception:
        pass
    video = VideoFileClip(video_path)
    audio = AudioFileClip(audio_path)
    end_sec = min(video.duration, audio.duration)
    video = video.with_audio(audio) if hasattr(video, 'with_audio') else video.set_audio(audio)
    video = video.subclipped(0, end_sec) if hasattr(video, 'subclipped') else video.subclip(0, end_sec)
    target_h, target_w = 720, 405
    if video.w > video.h:
        video = video.resize((target_w, target_h))
    print(f'[COMPOSE] Clips cargados ({time.time()-t0:.1f}s). Creando subtitulos...', flush=True)
    lines = [x.strip() for x in script.split('.') if x.strip()] or [script]
    n = len(lines)
    step = video.duration / n if n else video.duration
    txt_clips = []
    for i, line in enumerate(lines):
        try:
            tc = TextClip(line, fontsize=28, color='white', stroke_color='black', stroke_width=2)
            tc = tc.set_start(i * step).set_duration(step).set_position(('center', 'bottom'))
            txt_clips.append(tc)
        except Exception:
            pass
    if txt_clips:
        try:
            video = CompositeVideoClip([video] + txt_clips)
        except Exception:
            pass
        for tc in txt_clips:
            try:
                tc.close()
            except Exception:
                pass
        del txt_clips
    print('[COMPOSE] Escribiendo MP4 final (threads=1 para menos RAM en Colab)...', flush=True)
    video.write_videofile(out_path, fps=24, codec='libx264', audio_codec='aac', verbose=False, logger=None, threads=1)
    video.close()
    audio.close()
    del video
    gc.collect()
    print(f'[COMPOSE] Listo en {time.time()-t0:.1f}s', flush=True)
    return out_path

In [9]:
# Celda 6: No precargamos modelos para ahorrar RAM. Se cargaran al generar (voz primero, luego video).
tts_model = None
print('Listo. Los modelos se descargaran/cargaran al pulsar Generar:', flush=True)
print('  1) Primero el modelo de VOZ (TTS) -> genera el audio.', flush=True)
print('  2) Se libera la voz para ahorrar RAM.', flush=True)
print('  3) Luego el modelo de VIDEO (Wan) -> genera las imagenes.', flush=True)
print('Revisa la consola de logs en la interfaz para ver cada paso.', flush=True)

Listo. Los modelos se descargaran/cargaran al pulsar Generar:
  1) Primero el modelo de VOZ (TTS) -> genera el audio.
  2) Se libera la voz para ahorrar RAM.
  3) Luego el modelo de VIDEO (Wan) -> genera las imagenes.
Revisa la consola de logs en la interfaz para ver cada paso.


In [10]:
# Celda 7: Pipeline completo. run_pipeline_gen hace yield tras cada paso para que la consola se actualice en vivo.
video_pipeline = None

class TeeOut:
    def __init__(self, real, buf):
        self.real, self.buf = real, buf
    def write(self, s):
        self.real.write(s); self.buf.write(s)
    def flush(self):
        self.real.flush(); self.buf.flush()
    def close(self):
        self.flush()
    def isatty(self):
        return getattr(self.real, 'isatty', lambda: False)()

def run_pipeline(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    for path, msg, _ in run_pipeline_gen(texto_acertijo, prompt_video, speaker):
        pass
    return path, msg

def run_pipeline_gen(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    global tts_model, video_pipeline
    import time
    import gc
    import sys
    from io import StringIO
    log_buf = StringIO()
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout = TeeOut(old_stdout, log_buf)
    sys.stderr = TeeOut(old_stderr, log_buf)
    t0 = time.time()
    try:
        print('='*60, flush=True)
        print('[PIPELINE] INICIO. Consola activa: se actualiza en cada paso.', flush=True)
        print('='*60, flush=True)
        yield None, 'Iniciando pipeline...', log_buf.getvalue()
        if not texto_acertijo or not texto_acertijo.strip():
            yield None, 'Escribe el texto del acertijo.', log_buf.getvalue()
            return
        script = sanitize_script(texto_acertijo)
        if not script:
            yield None, 'Texto no valido.', log_buf.getvalue()
            return
        prompt_video = sanitize_video_prompt(prompt_video) if prompt_video else 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'
        base = OUTPUT_DIR / f'clip_{int(time.time())}'
        base.parent.mkdir(parents=True, exist_ok=True)
        audio_path = str(base) + '_audio.wav'
        video_path = str(base) + '_video.mp4'
        final_path = str(base) + '_final.mp4'
        duration_audio = 0.0
        try:
            yield None, 'Paso 1/3 - VOZ (cargando/generando audio)...', log_buf.getvalue()
            if tts_model is None:
                print('[PIPELINE] Paso 1/3 - VOZ: Cargando modelo de VOZ (TTS)...', flush=True)
                tts_model = get_tts_model()
            else:
                print('[PIPELINE] Paso 1/3 - VOZ: Generando audio...', flush=True)
            _, _, duration_audio = generate_voice(script, audio_path, speaker=speaker, model=tts_model)
            print(f'[PIPELINE] Paso 1/3 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            print('[PIPELINE] Liberando modelo de voz para liberar RAM...', flush=True)
            m = tts_model
            tts_model = None
            del m
            gc.collect()
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(1)
            print('[PIPELINE] RAM liberada. Cargando modelo de VIDEO en GPU.', flush=True)
            yield None, 'Paso 1/3 listo. Liberando voz.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en voz: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 2/3 - VIDEO (cargando/generando en GPU)...', log_buf.getvalue()
            if not torch.cuda.is_available():
                yield None, 'Error: No hay GPU. Runtime > Change runtime type > GPU (T4).', log_buf.getvalue()
                return
            if video_pipeline is None:
                gc.collect()
                torch.cuda.empty_cache()
                print('[PIPELINE] Paso 2/3 - VIDEO: Cargando modelo ModelScope en GPU...', flush=True)
                video_pipeline = get_video_pipeline()
            else:
                print('[PIPELINE] Paso 2/3 - VIDEO: Generando frames...', flush=True)
            generate_video(prompt_video, video_path, num_frames=24, pipeline=video_pipeline)
            print(f'[PIPELINE] Paso 2/3 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            yield None, 'Paso 2/3 listo.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en video: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 3/3 - Composicion final...', log_buf.getvalue()
            print('[PIPELINE] Paso 3/3 - COMPOSICION...', flush=True)
            compose_video(video_path, audio_path, script, final_path, duration_audio)
            print(f'[PIPELINE] FIN. Tiempo total: {time.time()-t0:.1f}s', flush=True)
            yield final_path, f'Listo en {time.time()-t0:.1f}s', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error al componer: {e}', log_buf.getvalue()
    except Exception as e:
        import traceback
        traceback.print_exc()
        yield None, str(e), log_buf.getvalue()
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr


In [11]:
# Celda 8: Interfaz Gradio. generar es un generador que hace yield en cada paso para que la Consola se actualice en vivo.
import gradio as gr

def generar(texto, prompt_video, speaker):
    for path, msg, logs in run_pipeline_gen(texto, prompt_video, speaker):
        yield path, msg, logs

demo = gr.Interface(
    fn=generar,
    inputs=[
        gr.Textbox(label='Texto del acertijo', placeholder='Ej: Que cosa tiene dientes y no muerde? La sierra.'),
        gr.Textbox(label='Descripcion del video (opcional)', value='Mysterious atmosphere, riddle, enigma, dark mood, cinematic, vertical', lines=2),
        gr.Dropdown(choices=['Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee'], value='Ryan', label='Voz'),
    ],
    outputs=[
        gr.Video(label='Video'),
        gr.Textbox(label='Estado'),
        gr.Textbox(label='Consola (logs)', lines=18, max_lines=30),
    ],
    title='Creador de Clips - Acertijos IG',
    description='Genera un video con voz y subtitulos. Suele tardar 6-11 min. Revisa la consola debajo si hay error.',
)

In [12]:
# Celda 8b: Descargar modelos ANTES de lanzar la interfaz (ejecutar una vez; luego los pesos quedan en cache)
# Asi la primera generacion no se bloquea por la descarga. Opcional pero recomendado en Colab.
import gc
print('Pre-descarga: modelo de VOZ (Qwen TTS)...', flush=True)
_m = get_tts_model()
del _m
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Pre-descarga: modelo de VIDEO (ModelScope 1.7B)...', flush=True)
_p = get_video_pipeline()
del _p
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Listo. Modelos descargados en cache. Al pulsar Generar se cargaran desde disco.', flush=True)

Pre-descarga: modelo de VOZ (Qwen TTS)...
[TTS] Estamos en: descargando/cargando modelo de VOZ (Qwen3-TTS). Este modelo narra el acertijo.

********
********
 
[TTS] Dispositivo: cpu (GPU = menos RAM, mas rapido). Descargando pesos si es la primera vez...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[TTS] Modelo de voz listo en 23.5s
Pre-descarga: modelo de VIDEO (ModelScope 1.7B)...


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


...

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


[VIDEO] Pipeline en GPU (18.3s). VRAM: 3.7 GB


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `TextToVideoSDPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


Listo. Modelos descargados en cache. Al pulsar Generar se cargaran desde disco.


In [13]:
# Celda 9: Lanzar interfaz (solo link publico, no embebido en Colab)
demo.launch(share=True, inline=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d84cde2061ee55776.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
